# ML_LR_comparison — Colab Runner

動態學習率排程對比 + Grad-CAM 視覺化。依序執行下列 cell 即可在 Colab GPU 上跑完 5 組實驗並產出對比圖。

**前置：** Runtime → Change runtime type → 選 GPU (T4 / V100 / A100 皆支援，會自動套對應 profile)。

## 1. 確認 GPU

In [ ]:
!nvidia-smi

## 2. 掛載 Google Drive（用來持久化 logs / checkpoints）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ML_LR_comparison'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive output root:', DRIVE_ROOT)

## 3. Clone 本 repo + 安裝套件

In [ ]:
%cd /content
!rm -rf ML_LR_comparison
!git clone https://github.com/eric20041027/ML_LR_comparison.git
%cd /content/ML_LR_comparison
!pip install -q -r requirements.txt

## 4. 下載並解壓 Tiny-ImageNet 到 `/content/data`（Colab 暫存區，速度快）

In [ ]:
!python -m scripts.download_data --data-dir /content/data

## 5. 跑 5 組實驗

輸出寫到 Drive 以便斷線後保留。預設 `--epochs 20`；視時間調整。

**自動偵測 GPU**：A100 → `a100` profile（大 batch + AMP + TF32 + 線性 LR scaling）；其它 (T4/V100/...) → `t4` profile。同一 session 中 5 組實驗使用同一 profile，對比依然公平。

In [ ]:
import torch

DATA_ROOT = '/content/data/tiny-imagenet-200'
OUTPUT_DIR = f'{DRIVE_ROOT}/experiments'
EPOCHS = 20  # 改小 (e.g. 5) 先做 smoke test

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
PROFILE = 'a100' if 'A100' in gpu_name else 't4'
print(f'Detected GPU: {gpu_name}  ->  profile = {PROFILE}')

!python -m scripts.run_all --data-root {DATA_ROOT} --output-dir {OUTPUT_DIR} --epochs {EPOCHS} --profile {PROFILE}

## 6. 量化視覺化：LR 曲線 + Loss/Acc 曲線

In [ ]:
!python -m src.plot_lr --experiments-dir {OUTPUT_DIR} --out {OUTPUT_DIR}/lr_curves.png
!python -m src.plot_curves --experiments-dir {OUTPUT_DIR} --out {OUTPUT_DIR}/curves.png

from IPython.display import Image, display
display(Image(f'{OUTPUT_DIR}/lr_curves.png'))
display(Image(f'{OUTPUT_DIR}/curves.png'))

## 7. 質化視覺化：Grad-CAM 對比圖（5 排程 × 早/中/晚期）

In [ ]:
!python -m src.gradcam_viz \
    --experiments-dir {OUTPUT_DIR} \
    --data-root {DATA_ROOT} \
    --out {OUTPUT_DIR}/grad_cam_grid.png

from IPython.display import Image, display
display(Image(f'{OUTPUT_DIR}/grad_cam_grid.png'))

## 8. (可選) 在 Colab 內看 TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $OUTPUT_DIR